# Runbook chạy dự án trên Google Colab

Notebook này **chỉ để chạy trên Colab**. Nó clone repo, sinh artifacts, sao lưu Drive và mở tunnel.

## Làm 3 việc này TRƯỚC khi Run all

1. **Runtime → Change runtime type → T4 GPU → Save.** `config.py` đọc `DEVICE` lúc import, nên bật GPU sau khi đã chạy thì phải Restart runtime.
2. **Sửa dòng `REPO` ở cell dưới** thành địa chỉ repo của bạn.
3. Nếu repo là **private**, `git clone` sẽ hỏi user/token — dùng Personal Access Token làm mật khẩu.

## Về thời gian

| Bước | Thời gian |
|---|---|
| Cài thư viện | 2–4 phút |
| Sinh artifacts (`all`) | ~15–25 phút trên T4 |
| Build React + nạp 4 mô hình | 3–6 phút (lần đầu tải Qwen ~3 GB) |

> ⚠️ Phiên Colab ngắt là link tunnel chết. Làm demo và **chụp ảnh màn hình ngay trong phiên đó**.
> Đừng để tab idle quá ~90 phút, Colab sẽ tự ngắt.


## 1. Kiểm tra GPU — chạy cell này trước tiên

Nếu in ra `cpu` thì bạn chưa bật T4. Quay lại Runtime → Change runtime type.

In [ ]:
!nvidia-smi -L || echo 'CHUA BAT GPU - Runtime > Change runtime type > T4 GPU'
!python --version

## 2. Clone repo

**Sửa dòng `REPO` bên dưới.** Chạy lại cell này lần hai sẽ tự `git pull` thay vì clone đè.

In [ ]:
import os, subprocess

REPO = "https://github.com/MMMAlonea04/AI_Web_Apps_Streamlit_React.git"
DEST = "/content/ai_web_apps"

def run(*cmd):
    print("$", " ".join(cmd), flush=True)
    subprocess.run(list(cmd), check=True)

if os.path.isdir(os.path.join(DEST, ".git")):
    run("git", "-C", DEST, "pull")
else:
    run("git", "clone", REPO, DEST)

os.chdir(DEST)          # Colab dung chung mot tien trinh, cwd giu cho cac cell sau
print("thu muc:", os.getcwd())
run("git", "log", "--oneline", "-1")
print(os.listdir())

## 3. Cài thư viện

Dùng `requirements-colab.txt` — **không** cài lại torch, vì Colab đã có bản khớp CUDA của nó.
Cài đè `requirements.txt` ở đây dễ làm hỏng CUDA và tốn vài phút tải vô ích.

In [ ]:
!pip install -q -r requirements-colab.txt
!python -c "import config, torch; print('DEVICE =', config.DEVICE, '| GPU =', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'khong co'); print('LLM   =', config.LLM_MODEL)"

## 4. Sinh artifacts — bước lâu nhất (~15–25 phút)

5 stage chạy theo thứ tự cố định: `data · classifier · detector · retrieval · rag`.

**Nếu lỗi ở một stage**, sửa xong chạy lại riêng stage đó, không cần chạy lại từ đầu:

```python
!python scripts/build_artifacts.py rag
```

In [ ]:
!python scripts/build_artifacts.py all

## 5. Sao lưu NGAY — `/content` mất khi hết phiên

Cell trên chạy xong thì làm việc này trước khi làm bất cứ thứ gì khác.
Bỏ qua là mất 20 phút và phải train lại.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/ai_web_apps && cp -r artifacts data/gallery /content/drive/MyDrive/ai_web_apps/
!du -sh /content/drive/MyDrive/ai_web_apps/*

## 6. Xem số đo trước khi chạy giao diện

Đây là **số thật để điền vào `MODEL_CARD.md`**. Đừng tự ước lượng.

In [ ]:
import json, pathlib

for p in sorted(pathlib.Path('artifacts').rglob('*metrics*.json')):
    d = json.loads(p.read_text(encoding='utf-8'))
    keep = {k: v for k, v in d.items() if k != 'history' and not isinstance(v, dict)}
    print(f'--- {p} ---')
    print(json.dumps(keep, indent=1, ensure_ascii=False))
    if 'text_to_image_precision@10' in d:
        print('  text->image P@10:', d['text_to_image_precision@10'])

## 7. Chạy cả hai giao diện + link công khai

Bước này build React, khởi động FastAPI (nạp cả 4 mô hình) và Streamlit, rồi mở Cloudflare Tunnel.
`serve.py` tự cài Node 22 nếu bản Node của Colab quá cũ cho Vite 8.

Mở link in ra trên điện thoại để kiểm tra, rồi **chụp ảnh màn hình** làm minh chứng cho báo cáo.

In [ ]:
!python scripts/serve.py all

## 8. Sau khi xong

1. Tải `artifacts/` và `data/gallery/` từ Drive về máy.
2. Chép vào repo, rồi:

```bash
git add artifacts data/gallery && git commit -m "artifacts do tren Colab T4" && git push
```

3. Điền số thật vào `MODEL_CARD.md`.

### Nếu gặp lỗi

| Hiện tượng | Cách xử lý |
|---|---|
| Tải model Hugging Face lỗi 429 | `from huggingface_hub import login; login()` bằng token miễn phí |
| `CUDA out of memory` | Restart runtime rồi chạy lại từ cell 3 với `!python scripts/build_artifacts.py retrieval rag` |
| `serve.py` báo thiếu mô hình | Xem `logs/api.log`, chạy lại stage tương ứng |
| Link tunnel không mở | Chạy lại `!python scripts/serve.py tunnel` — mỗi lần ra link mới |
| Colab báo hết quota GPU | Đổi sang CPU, chấp nhận số yếu hơn, và ghi rõ trong báo cáo là đo trên CPU |
